# Comparing clustering methods: why they disagree

Give six clustering methods the **same days** and ask each for the **same number of typical
periods**, and they will not agree. Each optimizes a *different idea of a good grouping*, so each
carves the data differently — and that flows straight through to the typical periods your model
sees.

This tutorial makes that visible on **12 hand-designed days**, small enough to follow every period
by eye. By the end you will be able to:

1. **predict** which methods group by *value* and which by *calendar position*;
2. **explain** why `kmeans`, `kmedoids` and `hierarchical` find the same partition here, while
   `kmaxoids` splits off the storm and `contiguous`/`averaging` ignore similarity altogether;
3. **weigh** what each grouping costs you, in both accuracy and runtime.

> **This tutorial isolates the grouping.** Clustering has a second half — how each group becomes
> one profile — and it is an independent lever that would otherwise contaminate every number
> below. So we hold it fixed at `representation="mean"` throughout and pass
> `preserve_column_means=False` so nothing is adjusted afterwards. Every difference you see is
> the *grouping* and nothing else. The other half is
> [Comparing representations](comparing_representations.ipynb).

## 1  The data: 12 designed days

The sample is built so the lesson is unambiguous. There are four kinds of day:

| archetype | solar | load | how many |
|---|---|---|---|
| **sunny** | bright midday peak | low | 5 |
| **cloudy** | dim | medium | 4 |
| **high-load** | dim | high | 2 |
| **storm** | none | **extreme** load peak | 1 |

Two design choices do all the work:

* the single **storm** day is a clear outlier — far from every other day in value space;
* the days are **interleaved in the calendar** (sunny, cloudy, high-load, …) so that *position on
  the calendar tells you nothing about a day's shape*.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio

import tsam
from tsam import ClusterConfig
from tsam.plot import compare_partitions

pio.renderers.default = "notebook_connected"

# 12 designed days, 6-hourly, two attributes (solar, load).
days = pd.read_csv("../data/comparison_days.csv", index_col=0, parse_dates=True)

# What each day "really" is, by construction — used only for labels and colors.
archetype = [
    "sunny",
    "cloudy",
    "high-load",
    "sunny",
    "cloudy",
    "storm",
    "cloudy",
    "sunny",
    "high-load",
    "sunny",
    "cloudy",
    "sunny",
]

# Held fixed for every run below so that only the grouping varies.
COMMON = {
    "n_clusters": 3,
    "period_duration": "1D",
    "preserve_column_means": False,
}

print(f"{days.shape[0]} timesteps = 12 days x 4 six-hourly steps")
days.head(8)

In [ ]:
long = days.reset_index(names="time").melt(
    id_vars="time", var_name="attribute", value_name="value"
)
fig = px.line(
    long,
    x="time",
    y="value",
    facet_row="attribute",
    title="The 12 designed days — note the storm spike in load (day 5)",
)
fig.update_yaxes(matches=None)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.show()

## 2  Run all six at k = 3

Ask each method for **3 typical days** and line up what comes back.

In [ ]:
methods = ["kmeans", "kmedoids", "hierarchical", "kmaxoids", "contiguous", "averaging"]

results = {}
for method in methods:
    np.random.seed(0)  # kmeans and kmaxoids start from random centers
    results[method] = tsam.aggregate(
        days, **COMMON, cluster=ClusterConfig(method=method, representation="mean")
    )

pd.DataFrame(
    {
        "weighted RMSE": {
            m: round(float(r.accuracy.weighted_rmse), 4) for m, r in results.items()
        }
    }
)

**The first three rows are identical to four decimal places.** That is not a coincidence and
not a rounding artifact: with the representation held fixed, an identical error means an
identical partition. `kmeans`, `kmedoids` and `hierarchical` put exactly the same days in exactly
the same groups.

The grid below shows why — and where the other three diverge. Each column is one original day,
each row a method, and the color is the group that day landed in. Cluster **ids are arbitrary**,
so the grid renumbers each row by order of first appearance; without that, identical groupings
could look different purely because they numbered their groups differently.

In [ ]:
compare_partitions(results, labels=archetype)

Six methods, but only **four distinct partitions**:

| methods | in words |
|---|---|
| **kmeans, kmedoids, hierarchical** | the three *value* groups; **storm absorbed** into high-load |
| **kmaxoids** | **storm isolated** as its own cluster; cloudy + high-load merged |
| **contiguous** | value-aware but **calendar-blocked**: storm stranded as a singleton between two runs |
| **averaging** | three **equal calendar blocks**; values ignored |

The rest of the tutorial explains why each arises. The three distance-minimizers agree here only
because the value groups are well separated; the interesting question is why the other three
diverge.

## 3  The map: the 12 days in feature space

A clustering method never sees day-shapes. It sees each day as a single **point** in an 8-D space
(2 attributes × 4 timesteps) after normalization, and applies a rule for covering those points
with three centers. `plot.feature_space()` draws that space.

**Eight dimensions cannot be drawn, so the plot flattens them to two.** The tool for that is
*principal component analysis* (PCA): it finds the two directions along which the twelve days
differ most, and gives each day its position along them. Those two directions become the axes —
the *first* and *second principal component*, labelled `PC 1` and `PC 2`. Each is a blend of all
eight original values, so neither carries a unit and neither *is* "solar" or "load". Read the
result as a map: it tells you which days sit near each other and which sit far apart, not what
any single coordinate means. Flattening always loses something, so call `feature_space()`
without a `title` and its default one reports what share of the spread the two directions kept.

Two more things make it readable. Days that land on the same spot are drawn as **one marker with a
count** — five of these days are near-identical by construction, so labelling each separately
would produce an unreadable smudge. And the axes are locked to **equal aspect**, because a plot
whose whole subject is distance must not distort it.

In [ ]:
results["kmeans"].plot.feature_space(
    labels=archetype,
    title="Minimize distance — kmeans, kmedoids and hierarchical all land here",
)

Read the map: **sunny** days cluster on the right, **cloudy** top-center, **high-load** left, and
the lone **storm** sits far out on its own. The lines run from each group to the center it was
assigned to, and the stars mark those centers.

Note the storm's line: it reaches all the way across to the high-load center. **The storm has been
absorbed.** With three centers to spend, the cheapest way to minimize total distance is to cover
the three *dense* groups and let the lone outlier join the nearest one. The storm is one day out
of twelve — absorbing it barely moves the total.

## 4  Divergence I — minimize distance vs. maximize spread

`kmaxoids` inverts the objective. Instead of putting centers where the points are, it picks
centers that are **far apart**. The storm is the farthest point from everything else, so spending
a center on it *maximizes* spread:

In [ ]:
results["kmaxoids"].plot.feature_space(
    labels=archetype, title="Maximize spread — kmaxoids"
)

Same 12 points, a completely different covering. The storm now has a center to itself — and
notice the center markers changed shape. They are drawn as **rings, not stars**, because
`kmaxoids` centers are *real observed days*, whereas the `kmeans` centers above were computed
means that no day matches. Forced to cover everything else with the two centers that remain,
`kmaxoids` merges cloudy and high-load into one group.

That trade shows up in the error: `kmaxoids` scores worse than the distance-minimizers, because
reconstructing ten ordinary days got worse in exchange for capturing one extreme. Whether that is
a good deal depends on your model — if a single peak drives investment decisions, the storm may
matter more than average fit. [Extreme periods](../how-to/extreme_periods.ipynb) is the targeted
alternative: keep the storm *without* paying for it across the other ten days.

## 5  Divergence II — grouping by calendar position

The last two methods never look at value similarity at all; they group by **position on the
calendar**. Because our days are interleaved, this lands them far from the value-based partition.

* **averaging** cuts the timeline into three **equal blocks** (days 0–3, 4–7, 8–11), whatever sits
  inside;
* **contiguous** is value-*aware* but may only merge **adjacent** days, so it is forced into runs —
  here it strands the storm as a singleton block between two long runs.

Both keep calendar order; neither can put the scattered sunny days together.

In [ ]:
results["averaging"].plot.clusters_over_time(
    columns=["load"], title="Averaging — three equal calendar blocks, values ignored"
)

In [ ]:
results["contiguous"].plot.clusters_over_time(
    columns=["load"], title="Contiguous — value-aware, but only adjacent days may merge"
)

Both cost roughly **three times the error** of the similarity-based methods on this data. That is
the price of the calendar constraint, and it is worth being precise about who pays it: being
value-aware still helps — `contiguous` beats blind `averaging` — but not nearly enough to close
the gap to methods that may group any day with any other.

These two are for when calendar order *must* be preserved (a typical period that has to map onto a
contiguous season, for example), not for best fit. Their formulation is covered in
[Agglomerative clustering](../explanation/how-aggregation-works/02_clustering/02_agglomerative_clustering.ipynb)
and [Averaging](../explanation/how-aggregation-works/02_clustering/04_averaging.ipynb).

## 6  The consequence: reconstruction

"Different partition" is abstract; the reconstructed load makes it concrete. Each real day is
replaced by its group's profile and stitched back onto the calendar. Watch the storm (day 5) and
the interleaving:

In [ ]:
frames = [
    pd.DataFrame(
        {"time": days.index, "load": days["load"].values, "series": "original"}
    )
]
for method in ["kmeans", "kmaxoids", "averaging"]:
    series = results[method].reconstructed["load"]
    frames.append(
        pd.DataFrame({"time": series.index, "load": series.values, "series": method})
    )
px.line(
    pd.concat(frames),
    x="time",
    y="load",
    color="series",
    title="Load reconstruction — kmeans tracks the shape, kmaxoids reaches the storm, "
    "averaging is blocky",
).show()

**kmeans** follows the day-to-day shape but clips the storm peak; **kmaxoids** is the only one to
reach the storm's height, having kept a center there, but is coarser everywhere else;
**averaging** is blocky and value-blind, missing both the shape and the peak.

## 7  The other cost: runtime

Accuracy is only half the decision. The six methods differ enormously in what they cost to
*run* — and that gap, not fit, decides which ones you can still afford once a year of days replaces
twelve.

You might expect to settle it right here, by timing all six on these 12 days. **Don't** — it is the
one question these designed days cannot answer, and seeing why is the lesson. At 12 periods you
measure fixed overhead, not the algorithms, and the ranking that overhead produces is not merely
noisy but *inverted*:

* **`kmeans` looks slow** here, because scikit-learn restarts it ten times from different seeds and
  that setup dwarfs the real work at this size. On a year of days it is one of the *fastest*.
* **`kmedoids` looks cheap** here, and on a year of days it is the *slowest, by two orders of
  magnitude* — it solves an exact MILP whose size grows with the **square** of the period count, so
  it barely notices twelve periods and chokes on hundreds.

That is the exact trap a small-sample benchmark sets, sprung here on data built to make it obvious.
All these 12 days honestly tell you is the coarsest fact — the six are not interchangeable on cost,
and the three cheap ones (`averaging`, `hierarchical`, `contiguous`) are cheap by a wide margin.

For runtimes you can carry away — measured across dataset sizes, with the crossover where the
ranking flips and the levers that move it — see
**[How long will this take?](../how-to/runtime.ipynb)**.

## 8  Recap — which logic, when

| method | grouping logic | reach for it when… |
|---|---|---|
| **kmeans** | minimize distance to centroids | you want the best average fit, and it scales |
| **kmedoids** | minimize distance, centers are real days | centers must be observed periods, and the dataset is small |
| **hierarchical** | merge nearest groups bottom-up | you want a robust, deterministic, fast default |
| **kmaxoids** | maximize spread between centers | the spread matters more than average fit |
| **contiguous** | merge nearest *adjacent* groups | calendar order must be preserved |
| **averaging** | equal calendar blocks | you need the simplest possible baseline |

The headline: **what a method optimizes decides what it keeps.** Distance-minimizers protect the
average and quietly drop extremes; spread-maximizers do the reverse; calendar methods preserve
order at the expense of similarity.

Everything above held the representation fixed to isolate the grouping. In real use it is a
second, independent choice — and each method has its own *default*, so switching method silently
switches representation too unless you say otherwise. That trap is the subject of
[Choosing a method](choosing_a_method.ipynb).

**Where to go next**

- [Comparing representations](comparing_representations.ipynb) — the other half of the decision:
  what each group *becomes*, all six rules on one cluster.
- [Choosing a method](choosing_a_method.ipynb) — grouping and representation together, plus
  extremes and segmentation.
- [How long will this take?](../how-to/runtime.ipynb) — runtime across dataset sizes, and how to
  pick a method you can afford.
- [Clustering methods how-to](../how-to/clustering_methods.ipynb) — the task-oriented recipe.
- [Partitional clustering](../explanation/how-aggregation-works/02_clustering/01_partitional_clustering.ipynb) —
  how k-means and k-medoids are formulated and solved.
- [Extremal-prototype selection](../explanation/how-aggregation-works/02_clustering/03_extremal_prototype_selection.ipynb) —
  how k-maxoids maximizes spread instead.